In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import os
from datetime import datetime
sys.path.append(r"C:/Users/AnalistaJ/Documents/RGM/Desarrollo/project_root/")
ROOT_DIR = Path("C:/Users/AnalistaJ/Documents/RGM/Desarrollo/project_root")
from src.connection.to_sql import sp_vg_MSV, descargar_tabla, sp_inv
from src.data.map import map_almacen

In [2]:
dias_transcurridos = 8
dias_laborales = 26

exportar_trimestre = False

In [3]:
fecha_inicio = pd.to_datetime('2026-08-01')
fecha_final = pd.to_datetime('2026-08-10')

fecha_inicial_aa = fecha_inicio - pd.DateOffset(years=1)
fecha_final_aa = (fecha_final - pd.DateOffset(years=1) + pd.DateOffset(months=3)).replace(day=1) + pd.offsets.MonthEnd(0)

## CONSULTAR Y UNIR DATOS

In [4]:
df_act = sp_vg_MSV(fecha_inicio, fecha_final) #YY-MM-DD
#df_aa = sp_vg_MSV(fecha_inicial_aa, fecha_final_aa) #YY-MM-DD

c:\Users\AnalistaJ\Documents\RGM\Desarrollo\project_root\src\connection\to_sql.py:199: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn, params=[fecha_inicial, fecha_final]) #'02-07-2026'


In [5]:
df_aa = sp_vg_MSV(fecha_inicial_aa, fecha_final_aa) #YY-MM-DD

In [6]:
df_inv = sp_inv('TODOS')

c:\Users\AnalistaJ\Documents\RGM\Desarrollo\project_root\src\connection\to_sql.py:207: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn, params=[codigo, parametro,0,'2025-12-24',0])


In [7]:
conds = [
    df_inv['Minimo'] == 0,
    df_inv['Minimo'] == 9999
]
choices = ['Moda', 'Basico-Moda']
df_inv['basico_moda'] = np.select(conds, choices, default='Basico')

In [8]:
df_act = df_act[df_act['target'].isin(['Menudeo', 'Combinado'])]
df_act['codigo'] = (
    df_act['articulo'].astype(str)
    + df_act['subcuenta'].fillna('').astype(str)
)

df_aa = df_aa[df_aa['target'].isin(['Menudeo', 'Combinado'])]
df_aa['codigo'] = (
    df_aa['articulo'].astype(str)
    + df_aa['subcuenta'].fillna('').astype(str)
)

In [9]:
cols = ['mes', 'sucursal', 'categoria', 'rama', 'familia', 'codigo']

df_aa[cols] = df_aa[cols].fillna('SIN_DATO')
df_act[cols] = df_act[cols].fillna('SIN_DATO')

In [10]:
datos_art = pd.concat([
    df_act[['codigo','articulo','subcuenta','descripcion1','Color']]
        .drop_duplicates(subset=['codigo']),
    df_aa[['codigo','articulo','subcuenta','descripcion1','Color']]
        .drop_duplicates(subset=['codigo'])
]).drop_duplicates(subset=['codigo'])


In [11]:
df_group = df_act.groupby(['mes','sucursal','categoria','subcategoria','familia','codigo']).agg(
    {'VentaTotal_sin_Monedero': 'sum',
     'costototal': 'sum'}).reset_index()

df_group_aa = df_aa.groupby(['mes','sucursal','categoria','subcategoria','familia','codigo']).agg(
    {'VentaTotal_sin_Monedero': 'sum',
     'costototal': 'sum'}).reset_index() 

In [12]:
df_group_aa_mes_act = df_group_aa[df_group_aa['mes'].isin(df_group['mes'].unique())]

In [13]:
df_merge = df_group.merge(df_group_aa_mes_act, how='outer', on=['mes','sucursal','categoria','subcategoria','familia','codigo'], suffixes=('_act', '_aa')).fillna(0)

## COLUMNAS CALCULADAS

In [14]:
df_merge['margen_act'] = np.where(
    df_merge['VentaTotal_sin_Monedero_act'] != 0,
    (df_merge['VentaTotal_sin_Monedero_act'] - df_merge['costototal_act']) / df_merge['VentaTotal_sin_Monedero_act'],
    0
)
df_merge['margen_aa'] = np.where(
    df_merge['VentaTotal_sin_Monedero_aa'] != 0,
    (df_merge['VentaTotal_sin_Monedero_aa'] - df_merge['costototal_aa']) / df_merge['VentaTotal_sin_Monedero_aa'],
    0
)

In [15]:
df_merge['tendencia_venta'] = df_merge['VentaTotal_sin_Monedero_act'] / dias_transcurridos * dias_laborales

In [16]:
df_merge['crecimiento_venta'] = ((df_merge['tendencia_venta'] - df_merge['VentaTotal_sin_Monedero_aa']) / df_merge['VentaTotal_sin_Monedero_aa'].replace(0, np.nan)).fillna(0)
df_merge['diferencia_venta'] = df_merge['tendencia_venta'] - df_merge['VentaTotal_sin_Monedero_aa']
df_merge['dif_MC'] = df_merge['margen_act'] - df_merge['margen_aa']
df_merge['tendencia_costo'] = df_merge['costototal_act'] / dias_transcurridos * dias_laborales
df_merge['margen_act_$'] = df_merge['VentaTotal_sin_Monedero_act'] - df_merge['costototal_act']
df_merge['margen_tend_$'] = df_merge['tendencia_venta'] - df_merge['tendencia_costo']
df_merge['margen_aa_$'] = df_merge['VentaTotal_sin_Monedero_aa'] - df_merge['costototal_aa']
df_merge['dif_utilidad'] = df_merge['margen_act_$'] - df_merge['margen_aa_$']

## DIAS DE INVENTARIO

In [17]:
#Calculo con trimestre siguiente del AA
df_group_tri_aa = (
    df_aa[df_aa['mes'].isin(df_aa['mes'].unique()[-3:])].groupby(
        ['sucursal','categoria','subcategoria','familia','codigo'], as_index=False)
         .agg({'cantidadinventario': 'sum'})
)
df_group_tri_aa['minimov2'] = df_group_tri_aa['cantidadinventario'] / 3
df_group_tri_aa = df_group_tri_aa.drop(columns='cantidadinventario')

df_merge_tri_ven = df_merge.merge(df_group_tri_aa, how='left', on=['sucursal','categoria','subcategoria','familia','codigo']).fillna(0)

In [18]:
# Calculo con mínimo en tabla artalm
#df_planeador = descargar_tabla('ticplandiario')
df_artalm = descargar_tabla('artalm')

df_artalm['Almacen2'] = df_artalm['Almacen'].map(map_almacen)

c:\Users\AnalistaJ\Documents\RGM\Desarrollo\project_root\src\connection\to_sql.py:46: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [19]:
df_artalm_group = df_artalm.groupby(['Articulo','SubCuenta','Almacen2']).agg({'Minimo': 'sum'}).reset_index()
df_artalm_group['codigo'] = (
    df_artalm_group['Articulo'].astype(str)
    + df_artalm_group['SubCuenta'].fillna('').astype(str)
)

In [20]:
# Normalizar sucursales
df_merge_tri_ven['sucursal'] = df_merge_tri_ven['sucursal'].str.upper()
df_artalm_group['Almacen2'] = df_artalm_group['Almacen2'].str.upper()

# Agregar la columna Minimo
df_merge_tri_ven = df_merge_tri_ven.merge(
    df_artalm_group[['codigo', 'Almacen2', 'Minimo']],
    left_on=['codigo', 'sucursal'],
    right_on=['codigo', 'Almacen2'],
    how='left'
)

# Sólo reemplazar donde minimov2 sea 0
mask = df_merge_tri_ven['minimov2'] == 0
df_merge_tri_ven.loc[mask, 'minimov2'] = (
    df_merge_tri_ven.loc[mask, 'Minimo']
    .fillna(0)
)

# Eliminar columnas auxiliares
df_merge_tri_ven = df_merge_tri_ven.drop(columns=['Almacen2', 'Minimo'])

In [21]:
# Calculo de venta actual en tendencia
df_aux_dias_inv = df_act.groupby(['mes','sucursal','categoria','subcategoria','familia','codigo']).agg({'cantidadinventario': 'sum'}).reset_index()
df_aux_dias_inv['minimov2_aux'] = df_aux_dias_inv['cantidadinventario'] / dias_transcurridos * dias_laborales

In [22]:
# Normalizar sucursales
df_aux_dias_inv['sucursal'] = df_aux_dias_inv['sucursal'].str.upper()

# Agregar la columna minimov2 del dataframe auxiliar
df_merge_tri_ven = df_merge_tri_ven.merge(
    df_aux_dias_inv[['codigo', 'sucursal', 'minimov2_aux']],
    on=['codigo', 'sucursal'],
    how='left'
)

# Reemplazar únicamente donde minimov2 sea 0
mask = df_merge_tri_ven['minimov2'] == 0
df_merge_tri_ven.loc[mask, 'minimov2'] = (
    df_merge_tri_ven.loc[mask, 'minimov2_aux']
    .fillna(0)
)

# Eliminar la columna auxiliar
df_merge_tri_ven = df_merge_tri_ven.drop(columns='minimov2_aux')

## AGREGAR INVENTARIOS

In [23]:
df_inv['Almacen2'] = df_inv['Almacen'].str.strip().map(map_almacen)

In [24]:
df_inv['codigo'] = (
    df_inv['articulo'].astype(str)
    + df_inv['subcuenta'].fillna('').astype(str)
)

In [25]:
df_merge_inv = df_merge_tri_ven.merge(
    df_inv[['codigo', 'Almacen2', 'disponible']],
    left_on=['codigo', 'sucursal'],
    right_on=['codigo', 'Almacen2'],
    how='left'
)

df_merge_inv = df_merge_inv.rename(columns={'disponible': 'inv_une'})
df_merge_inv = df_merge_inv.drop(columns='Almacen2')

In [26]:
df_inv_aldis = (
    df_inv[df_inv['Almacen2'] == 'ALDIS']
    .groupby('codigo', as_index=False)['disponible']
    .sum()
    .rename(columns={'disponible': 'inv_aldis'})
)

df_merge_inv = df_merge_inv.merge(
    df_inv_aldis,
    on='codigo',
    how='left'
)

df_merge_inv = df_merge_inv.rename(columns={'disponible': 'inv_aldis'}).fillna(0)

In [27]:
df_merge_inv = df_merge_inv[df_merge_inv['sucursal'].isin(['CAMPECHE', 'CENTRO', 'CHARLY', 'KABAH', 'NORTE'])]

In [28]:
df_merge_inv['dias_inv_une'] = np.where(
    df_merge_inv['minimov2'] > 0,
    (df_merge_inv['inv_une'] / df_merge_inv['minimov2']) * 30,
    0
)

In [29]:
minimov2_codigo = (
    df_merge_inv
    .groupby('codigo')['minimov2']
    .sum()
    .rename('minimov2_total_codigo')
)

In [30]:
df_merge_inv = df_merge_inv.merge(
    minimov2_codigo,
    on='codigo',
    how='left',
    validate='many_to_one'
)

In [31]:
df_merge_inv['dias_inv_aldis'] = np.where(
    df_merge_inv['minimov2_total_codigo'] > 0,
    (df_merge_inv['inv_aldis'] / df_merge_inv['minimov2_total_codigo']) * 30,
    0
)

In [32]:
df_merge_info = df_merge_inv.merge(datos_art, how='left', on=['codigo'], validate='many_to_one')

In [33]:
df_merge_info.columns

Index(['mes', 'sucursal', 'categoria', 'subcategoria', 'familia', 'codigo',
       'VentaTotal_sin_Monedero_act', 'costototal_act',
       'VentaTotal_sin_Monedero_aa', 'costototal_aa', 'margen_act',
       'margen_aa', 'tendencia_venta', 'crecimiento_venta', 'diferencia_venta',
       'dif_MC', 'tendencia_costo', 'margen_act_$', 'margen_tend_$',
       'margen_aa_$', 'dif_utilidad', 'minimov2', 'inv_une', 'inv_aldis',
       'dias_inv_une', 'minimov2_total_codigo', 'dias_inv_aldis', 'articulo',
       'subcuenta', 'descripcion1', 'Color'],
      dtype='object')

In [34]:
df_b_m = df_inv[['codigo','Almacen2','basico_moda']].drop_duplicates(subset=['codigo','Almacen2']).rename(columns={'Almacen2': 'sucursal'})
df_merge_info = df_merge_info.merge(df_b_m, how='left', on=['codigo','sucursal'], validate='many_to_one')

In [35]:
df_merge_info = df_merge_info[['mes','sucursal','categoria','subcategoria','familia','codigo','articulo','subcuenta','descripcion1','Color','basico_moda','VentaTotal_sin_Monedero_act','tendencia_venta','VentaTotal_sin_Monedero_aa','crecimiento_venta','diferencia_venta','margen_act','dif_MC','margen_aa','costototal_act','costototal_aa','margen_act_$','margen_aa_$','margen_tend_$','dif_utilidad','minimov2','inv_une','dias_inv_une','inv_aldis','dias_inv_aldis']]
df_merge_just_info = df_merge_info[['mes','sucursal','categoria','subcategoria','familia','codigo','articulo','subcuenta','descripcion1','Color','basico_moda']]

## ORGANIZAR PARA EXPORTAR

In [36]:
dfs_por_sucursal = {
    sucursal: grupo.copy()
    for sucursal, grupo in df_merge_info.groupby('sucursal')
}

# Carpeta de salida
ruta_salida = ROOT_DIR / "data" / "reportes_sucursales"
ruta_salida.mkdir(parents=True, exist_ok=True)

for sucursal, df_sucursal in dfs_por_sucursal.items():
    
    archivo_excel = ruta_salida / f"{sucursal}.xlsx"
    
    with pd.ExcelWriter(archivo_excel, engine="openpyxl") as writer:
        df_sucursal.to_excel(
            writer,
            sheet_name="Todos",
            index=False
        )
        for categoria, df_categoria in df_sucursal.groupby('categoria'):
            
            nombre_hoja = str(categoria)[:30]
            
            df_categoria.to_excel(
                writer,
                sheet_name=nombre_hoja,
                index=False
            )

## SEGUNDO FORMATO

In [37]:
#df_tri_2['VentaTotal_sin_Monedero'].sum()

In [38]:
df_tri_2 = df_aa[df_aa['mes'].isin(df_aa['mes'].unique()[-3:])].groupby(['mes','sucursal','codigo'], as_index=False).agg({'VentaTotal_sin_Monedero':'sum','cantidadinventario': 'sum'})

In [39]:
df_pivot = df_tri_2.pivot_table(
        index=['sucursal', 'codigo'],
        columns='mes',
        values=['VentaTotal_sin_Monedero', 'cantidadinventario'],
        aggfunc='sum',      
        fill_value=0
    ).reset_index()

df_pivot.columns = [
    f'{col}_{mes}'
    for col, mes in df_pivot.columns
]
df_pivot = df_pivot.rename(columns={'sucursal_': 'sucursal', 'codigo_': 'codigo'})

df_pivot = df_pivot.reset_index().drop(columns=['index'])

In [40]:
df_union = pd.concat([
    df_act[['sucursal','categoria','subcategoria','familia','codigo','articulo','subcuenta','descripcion1','Color','minimo']].drop_duplicates(),
    df_aa[['sucursal','categoria','subcategoria','familia','codigo','articulo','subcuenta','descripcion1','Color','minimo']].drop_duplicates()
], ignore_index=True).drop_duplicates()

In [41]:
conds = [
    df_union['minimo'] == 0,
    df_union['minimo'] == 9999
]
choices = ['Moda', 'Basico-Moda']
df_union['basico_moda'] = np.select(conds, choices, default='Basico')
df_union.drop(columns=['minimo'], inplace=True)

In [42]:
df_venta_tri_aa = df_union.merge(df_pivot, how='right', on=['sucursal','codigo'], validate='many_to_one').drop_duplicates(subset=['sucursal', 'codigo'])#.to_excel(ruta_salida / "Reporte_detalle.xlsx", index=False)

## EXPORTAR SEGUNDO FORMATO

In [ ]:


dfs_por_sucursal_tri = {
    sucursal: grupo.copy()
    for sucursal, grupo in df_venta_tri_aa.groupby('sucursal')
}

# Carpeta de salida
ruta_salida = ROOT_DIR / "data" / "reportes_sucursales"
ruta_salida.mkdir(parents=True, exist_ok=True)
if exportar_trimestre:
    for sucursal, df_sucursal in dfs_por_sucursal_tri.items():
        
        archivo_excel = ruta_salida / f"{sucursal}_tri.xlsx"
        
        with pd.ExcelWriter(archivo_excel, engine="openpyxl") as writer:
            df_sucursal.to_excel(
                writer,
                sheet_name="Todos",
                index=False
            )
            for categoria, df_categoria in df_sucursal.groupby('categoria'):
                
                nombre_hoja = str(categoria)[:30]
                
                df_categoria.to_excel(
                    writer,
                    sheet_name=nombre_hoja,
                    index=False
                )